# Graphviz cell renderer

Returning a `Dot(...)` from a cell renders it as inline SVG. The DOT → SVG step runs **entirely on the JVM kernel** via a bundled Graphviz wasm artifact executed by [chasm](https://github.com/CharlieTap/chasm) — no JS evaluation in the notebook frontend, no CDN.

Only the `dot` layout engine is bundled in this release; other engines (`neato`, `twopi`, ...) raise a clear `GraphvizException` until `wasm-build/` produces an artifact linking the `neato_layout` plugin.

**Prerequisite:** run `./gradlew :kotlin-notebook:publishToMavenLocal` from the repo root, then restart the kernel.

In [ ]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

## Simple graph

`Dot`, `DotEngine`, `DotOptions`, and `renderDot` are auto-imported by the integration's `onLoaded` hook — no explicit `import` lines needed in cells.

In [ ]:
Dot("""
    digraph G {
        rankdir=LR
        Input -> Dense1 -> Dense2 -> Output
        Dense2 -> Loss
    }
""".trimIndent())

## With options

Width and max-height flow into a wrapper `<div>` around the SVG. Engine other than `DOT` will raise `GraphvizException` (only `dot` is currently bundled).

In [ ]:
renderDot("""
    digraph DAG {
        node [shape=box, style=rounded]
        edge [color=gray]
        x [label="input"]
        y [label="output"]
        x -> conv1 -> bn1 -> relu1 -> conv2 -> bn2 -> y
        x -> y [style=dashed, label="residual"]
    }
""".trimIndent()) {
    width = "640px"
    maxHeight = "480px"
}